In [34]:
import os, json, random
import numpy as np
import pandas as pd
import math
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F

In [35]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [36]:
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [37]:
BATCH_SIZE = 512

## Load Data

In [38]:
data = torch.load("/kaggle/input/datasets/teamcatalyst/initial-dataset/torch_tensor_words.pt", map_location="cpu")

In [39]:
title_ids   = data["title_ids"]
title_mask  = data["title_mask"]
tag_ids     = data["tag_ids"]
tag_mask    = data["tag_mask"]

member_text_ids  = data["member_text_ids"]
member_text_mask = data["member_text_mask"]

hard_ids   = data["hard_ids"]
hard_mask  = data["hard_mask"]
soft_ids   = data["soft_ids"]
soft_mask  = data["soft_mask"]
int_ids    = data["int_ids"]
int_mask   = data["int_mask"]

In [40]:
with open("/kaggle/input/datasets/teamcatalyst/initial-dataset/word2id.json", "r") as f:
    word2id = json.load(f)
with open("/kaggle/input/datasets/teamcatalyst/initial-dataset/label2id.json", "r") as f:
    label2id = json.load(f)

In [41]:
member_ids_df  = pd.read_csv("/kaggle/input/datasets/teamcatalyst/initial-dataset/member_ids.csv")
project_ids_df = pd.read_csv("/kaggle/input/datasets/teamcatalyst/initial-dataset/project_ids.csv")

member_ids_list  = member_ids_df["member_id"].astype(int).tolist()
project_ids_list = project_ids_df["project_id"].astype(int).tolist()

full_member_id2idx  = {mid: i for i, mid in enumerate(member_ids_list)}
full_project_id2idx = {pid: i for i, pid in enumerate(project_ids_list)}

num_members_all  = len(member_ids_list)
num_projects_all = len(project_ids_list)

print("Full members:", num_members_all, "| Full projects:", num_projects_all)

Full members: 38462 | Full projects: 18569


In [42]:
assert member_text_ids.size(0) == num_members_all
assert hard_ids.size(0) == num_members_all
assert soft_ids.size(0) == num_members_all
assert int_ids.size(0) == num_members_all
assert title_ids.size(0) == num_projects_all
assert tag_ids.size(0) == num_projects_all

In [43]:
train_edges = pd.read_csv("/kaggle/input/datasets/teamcatalyst/one-hop-paper-data/train_edges.csv")
test_edges  = pd.read_csv("/kaggle/input/datasets/teamcatalyst/one-hop-paper-data/test_edges.csv")

for df in (train_edges, test_edges):
    df["member_id"]  = df["member_id"].astype(int)
    df["project_id"] = df["project_id"].astype(int)

print("Edges (raw):", train_edges.shape, test_edges.shape)

Edges (raw): (53242, 2) (13311, 2)


In [44]:
# Map to FULL indices
train_m_full = train_edges["member_id"].map(full_member_id2idx)
train_p_full = train_edges["project_id"].map(full_project_id2idx)
valid_train_full = train_m_full.notna() & train_p_full.notna()

train_members_full  = train_m_full[valid_train_full].astype(int).to_numpy()
train_projects_full = train_p_full[valid_train_full].astype(int).to_numpy()
print("Mapped TRAIN pairs (full space):", train_members_full.shape, train_projects_full.shape)
print("Dropped TRAIN edges (missing from full ids lists):", int((~valid_train_full).sum()))

Mapped TRAIN pairs (full space): (53242,) (53242,)
Dropped TRAIN edges (missing from full ids lists): 0


In [45]:
test_m_full = test_edges["member_id"].map(full_member_id2idx)
test_p_full = test_edges["project_id"].map(full_project_id2idx)
valid_test_full = test_m_full.notna() & test_p_full.notna()

test_members_full  = test_m_full[valid_test_full].astype(int).to_numpy()
test_projects_full = test_p_full[valid_test_full].astype(int).to_numpy()
print("Mapped TEST pairs (full space):", test_members_full.shape, test_projects_full.shape)
print("Dropped TEST edges (missing from full ids lists):", int((~valid_test_full).sum()))

Mapped TEST pairs (full space): (13311,) (13311,)
Dropped TEST edges (missing from full ids lists): 0


In [46]:
# 3) Warm/cold masks should be based on TRAIN EDGES in FULL space
warm_member_mask = np.zeros(num_members_all, dtype=bool)
warm_project_mask = np.zeros(num_projects_all, dtype=bool)
warm_member_mask[np.unique(train_members_full)] = True
warm_project_mask[np.unique(train_projects_full)] = True

print("Warm members:", int(warm_member_mask.sum()), "Cold members:", int((~warm_member_mask).sum()))
print("Warm projects:", int(warm_project_mask.sum()), "Cold projects:", int((~warm_project_mask).sum()))

Warm members: 33621 Cold members: 4841
Warm projects: 18487 Cold projects: 82


## One-Hop Classed (for cold-start)

In [47]:
class AdditiveAttentionEq6(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.b = nn.Linear(dim, dim, bias=True)     # b_w x + c_w
        self.a = nn.Parameter(torch.empty(dim))     # a_w
        nn.init.xavier_uniform_(self.b.weight)
        nn.init.zeros_(self.b.bias)
        nn.init.normal_(self.a, std=0.02)

    def forward(self, X: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.bool()
        B, L, D = X.shape

        H = torch.tanh(self.b(X))                  # [B,L,D]
        scores = torch.matmul(H, self.a)           # [B,L]

        valid_row = mask.any(dim=1)                # [B]
        out = torch.zeros((B, D), device=X.device, dtype=X.dtype)
        if valid_row.any():
            scores_v = scores[valid_row]
            mask_v   = mask[valid_row]
            neg_inf = torch.finfo(scores_v.dtype).min
            scores_v = scores_v.masked_fill(~mask_v, neg_inf)
            alpha = torch.softmax(scores_v, dim=1)
            out[valid_row] = torch.bmm(alpha.unsqueeze(1), X[valid_row]).squeeze(1)
        return out

In [48]:
def ensure_nonempty_sequence(ids_b, mask_b, unk_id=1):
    empty = ~mask_b.any(dim=1)
    if empty.any():
        ids_b = ids_b.clone()
        mask_b = mask_b.clone()
        ids_b[empty, 0] = unk_id
        mask_b[empty, 0] = True
    return ids_b, mask_b

In [49]:
class Eq5MultiHeadSelfAttention(nn.Module):
    def __init__(self, d: int, n_heads: int):
        super().__init__()
        self.d = d
        self.n_heads = n_heads
        self.w1 = nn.Parameter(torch.empty(n_heads, d, d))
        self.w2 = nn.Parameter(torch.empty(n_heads, d, d))
        nn.init.xavier_uniform_(self.w1)
        nn.init.xavier_uniform_(self.w2)

    def forward(self, E: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.bool()
        B, L, D = E.shape
        neg_inf = torch.finfo(E.dtype).min

        head_outs = []
        for k in range(self.n_heads):
            W1 = self.w1[k]  # [D,D]
            W2 = self.w2[k]  # [D,D]

            EW1 = torch.matmul(E, W1)                         # [B,L,D]
            scores = torch.matmul(EW1, E.transpose(1, 2))      # [B,L,L]

            key_mask = mask.unsqueeze(1).expand(B, L, L)       # mask keys j
            scores = scores.masked_fill(~key_mask, neg_inf)

            A = torch.softmax(scores, dim=2)                   # over j
            ctx = torch.matmul(A, E)                           # [B,L,D]
            h_k = torch.matmul(ctx, W2.transpose(0, 1))         # [B,L,D]
            head_outs.append(h_k)

        H = torch.cat(head_outs, dim=2)                         # [B,L,n_heads*D]
        H = H.masked_fill(~mask.unsqueeze(-1), 0.0)              # zero padded queries i
        return H

In [50]:
class Step1VariantEncoder(nn.Module):
    """
    Variant: task title uses Eq(5)+Eq(6), dev text ALSO uses Eq(5)+Eq(6),
    labels use Eq(6).
    """
    def __init__(self, word_vocab_size: int, label_vocab_size: int,
                 d_word: int = 128, d_label: int = 128, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.word_emb  = nn.Embedding(word_vocab_size, d_word, padding_idx=0)
        self.label_emb = nn.Embedding(label_vocab_size, d_label, padding_idx=0)
        self.dropout = nn.Dropout(dropout)

        self.eq5_words = Eq5MultiHeadSelfAttention(d=d_word, n_heads=n_heads)

        self.word_pool  = AdditiveAttentionEq6(dim=n_heads * d_word)  # pools Eq(5) outputs
        self.label_pool = AdditiveAttentionEq6(dim=d_label)

        self.n_heads = n_heads
        self.d_word = d_word
        self.d_label = d_label

    def encode_task(self, title_ids, title_mask, tag_ids, tag_mask):
        title_ids, title_mask = ensure_nonempty_sequence(title_ids, title_mask, unk_id=1)

        E = self.dropout(self.word_emb(title_ids))          # [B,L,Dw]
        H = self.eq5_words(E, title_mask)                   # [B,L,nH*Dw]
        vtitle = self.word_pool(H, title_mask)              # [B,nH*Dw]

        Lemb = self.dropout(self.label_emb(tag_ids))        # [B,Lt,Dl]
        vlabel = self.label_pool(Lemb, tag_mask)            # [B,Dl]

        return torch.cat([vtitle, vlabel], dim=1)           # [B, nH*Dw + Dl]

    def encode_developer(self, text_ids, text_mask, devlbl_ids, devlbl_mask):
        text_ids, text_mask = ensure_nonempty_sequence(text_ids, text_mask, unk_id=1)

        E = self.dropout(self.word_emb(text_ids))           # [B,L,Dw]
        H = self.eq5_words(E, text_mask)                    # [B,L,nH*Dw]
        utext = self.word_pool(H, text_mask)                # [B,nH*Dw]

        Lemb = self.dropout(self.label_emb(devlbl_ids))     # [B,Ll,Dl]
        ulabel = self.label_pool(Lemb, devlbl_mask)         # [B,Dl]

        return torch.cat([utext, ulabel], dim=1)            # [B, nH*Dw + Dl]

In [51]:
class OneHopCsdRecVariant(nn.Module):
    def __init__(self, word_vocab_size, label_vocab_size,
                 num_devs, num_tasks,
                 d_word=128, d_label=128, n_heads=4,
                 d_id=64, common_dim=256, dropout=0.1):
        super().__init__()
        self.d_id = d_id

        self.text = Step1VariantEncoder(
            word_vocab_size, label_vocab_size,
            d_word=d_word, d_label=d_label, n_heads=n_heads, dropout=dropout
        )

        self.dev_id  = nn.Embedding(num_devs,  d_id)
        self.task_id = nn.Embedding(num_tasks, d_id)

        task_sem_dim = (n_heads * d_word) + d_label
        dev_sem_dim  = (n_heads * d_word) + d_label  # text + labels

        self.dev_proj  = nn.Linear(d_id + dev_sem_dim,  common_dim)
        self.task_proj = nn.Linear(d_id + task_sem_dim, common_dim)

    def score(self, dev_idx, task_idx,
              # developer text + merged labels
              text_ids, text_mask, devlbl_ids, devlbl_mask,
              # task title+tags
              title_ids, title_mask, tag_ids, tag_mask,
              dev_known_mask=None, task_known_mask=None):

        # semantic embeddings
        u_t = self.text.encode_developer(text_ids, text_mask, devlbl_ids, devlbl_mask)
        v_t = self.text.encode_task(title_ids, title_mask, tag_ids, tag_mask)

        # ----- ID gating (cold-start) -----
        if dev_known_mask is None:
            u_id = self.dev_id(dev_idx)
        else:
            u_id = torch.zeros(
                (dev_idx.size(0), self.d_id),
                device=dev_idx.device,
                dtype=self.dev_id.weight.dtype
            )
            if dev_known_mask.any():
                u_id[dev_known_mask] = self.dev_id(dev_idx[dev_known_mask])

        if task_known_mask is None:
            v_id = self.task_id(task_idx)
        else:
            v_id = torch.zeros(
                (task_idx.size(0), self.d_id),
                device=task_idx.device,
                dtype=self.task_id.weight.dtype
            )
            if task_known_mask.any():
                v_id[task_known_mask] = self.task_id(task_idx[task_known_mask])

        # ----- project to common space + dot product -----
        z_u = self.dev_proj(torch.cat([u_id, u_t], dim=1))
        z_v = self.task_proj(torch.cat([v_id, v_t], dim=1))
        return (z_u * z_v).sum(dim=1)

In [52]:
def merge_dev_labels(h_ids_b, h_mask_b, s_ids_b, s_mask_b, i_ids_b, i_mask_b):
    devlbl_ids  = torch.cat([h_ids_b, s_ids_b, i_ids_b], dim=1)   # [B, Lh+Ls+Li]
    devlbl_mask = torch.cat([h_mask_b, s_mask_b, i_mask_b], dim=1)
    return devlbl_ids, devlbl_mask

In [53]:
onehop_model = OneHopCsdRecVariant(
    word_vocab_size=len(word2id),
    label_vocab_size=len(label2id),
    num_devs=num_members_all,
    num_tasks=num_projects_all,
    d_word=128,
    d_label=128,
    n_heads=4,
    d_id=64,
    common_dim=256,
    dropout=0.1
).to(device)

In [54]:
state = torch.load(
    "/kaggle/input/datasets/teamcatalyst/one-hop-paper-data/trained_one_hop_weights.pt",
    map_location="cpu"
)
onehop_model.load_state_dict(state, strict=True)
onehop_model.eval()

OneHopCsdRecVariant(
  (text): Step1VariantEncoder(
    (word_emb): Embedding(29261, 128, padding_idx=0)
    (label_emb): Embedding(4791, 128, padding_idx=0)
    (dropout): Dropout(p=0.1, inplace=False)
    (eq5_words): Eq5MultiHeadSelfAttention()
    (word_pool): AdditiveAttentionEq6(
      (b): Linear(in_features=512, out_features=512, bias=True)
    )
    (label_pool): AdditiveAttentionEq6(
      (b): Linear(in_features=128, out_features=128, bias=True)
    )
  )
  (dev_id): Embedding(38462, 64)
  (task_id): Embedding(18569, 64)
  (dev_proj): Linear(in_features=704, out_features=256, bias=True)
  (task_proj): Linear(in_features=704, out_features=256, bias=True)
)

In [55]:
# 6) Build base embeddings for ALL nodes (warm+cold)
#    - warm => uses ID embedding + semantics
#    - cold => ID embedding = zeros, keep semantics
# =========================
@torch.no_grad()
def build_all_member_base_embeddings():
    out_chunks = []
    for start in tqdm(range(0, num_members_all, BATCH_SIZE), desc="Build member base emb"):
        idx = np.arange(start, min(start + BATCH_SIZE, num_members_all))
        idx_t = torch.tensor(idx, dtype=torch.long, device=device)

        mt = member_text_ids[idx].to(device, non_blocking=True)
        mm = member_text_mask[idx].to(device, non_blocking=True)

        hi = hard_ids[idx].to(device, non_blocking=True); hm = hard_mask[idx].to(device, non_blocking=True)
        si = soft_ids[idx].to(device, non_blocking=True); sm = soft_mask[idx].to(device, non_blocking=True)
        ii = int_ids[idx].to(device, non_blocking=True);  im = int_mask[idx].to(device, non_blocking=True)

        devlbl_ids_b, devlbl_mask_b = merge_dev_labels(hi, hm, si, sm, ii, im)

        # semantic
        u_t = onehop_model.text.encode_developer(mt, mm, devlbl_ids_b, devlbl_mask_b)

        # ID gating: warm => id emb, cold => zeros
        known = torch.tensor(warm_member_mask[idx], dtype=torch.bool, device=device)
        u_id = torch.zeros((len(idx), onehop_model.d_id), device=device, dtype=onehop_model.dev_id.weight.dtype)
        if known.any():
            u_id[known] = onehop_model.dev_id(idx_t[known])

        z_u = onehop_model.dev_proj(torch.cat([u_id, u_t], dim=1))
        out_chunks.append(z_u.cpu())

    return torch.cat(out_chunks, dim=0).numpy()

@torch.no_grad()
def build_all_project_base_embeddings():
    out_chunks = []
    for start in tqdm(range(0, num_projects_all, BATCH_SIZE), desc="Build project base emb"):
        idx = np.arange(start, min(start + BATCH_SIZE, num_projects_all))
        idx_t = torch.tensor(idx, dtype=torch.long, device=device)

        ti = title_ids[idx].to(device, non_blocking=True)
        tm = title_mask[idx].to(device, non_blocking=True)
        gi = tag_ids[idx].to(device, non_blocking=True)
        gm = tag_mask[idx].to(device, non_blocking=True)

        v_t = onehop_model.text.encode_task(ti, tm, gi, gm)

        known = torch.tensor(warm_project_mask[idx], dtype=torch.bool, device=device)
        v_id = torch.zeros((len(idx), onehop_model.d_id), device=device, dtype=onehop_model.task_id.weight.dtype)
        if known.any():
            v_id[known] = onehop_model.task_id(idx_t[known])

        z_v = onehop_model.task_proj(torch.cat([v_id, v_t], dim=1))
        out_chunks.append(z_v.cpu())

    return torch.cat(out_chunks, dim=0).numpy()

In [56]:
member_base_emb  = build_all_member_base_embeddings().astype(np.float32)   # [num_members_all, common_dim]
project_base_emb = build_all_project_base_embeddings().astype(np.float32)  # [num_projects_all, common_dim]

print("member_base_emb:", member_base_emb.shape)
print("project_base_emb:", project_base_emb.shape)

# Save base embeddings aligned to FULL indices
np.save("/kaggle/working/member_base_embeddings_all.npy", member_base_emb)
np.save("/kaggle/working/project_base_embeddings_all.npy", project_base_emb)

# Save ID lists aligned with rows (full space)
pd.DataFrame({"member_id": member_ids_list}).to_csv("/kaggle/working/member_ids_all.csv", index=False)
pd.DataFrame({"project_id": project_ids_list}).to_csv("/kaggle/working/project_ids_all.csv", index=False)

Build project base emb: 100%|██████████| 37/37 [00:00<00:00, 103.90it/s]


member_base_emb: (38462, 256)
project_base_emb: (18569, 256)


In [57]:
# Also keep full-index train/test arrays for next two-hop step
np.save("/kaggle/working/train_members_full.npy", train_members_full)
np.save("/kaggle/working/train_projects_full.npy", train_projects_full)
np.save("/kaggle/working/test_members_full.npy", test_members_full)
np.save("/kaggle/working/test_projects_full.npy", test_projects_full)

## Prepare Neighbors

In [58]:
U2V = [set() for _ in range(num_members_all)]
V2U = [set() for _ in range(num_projects_all)]

for u, v in zip(train_members_full, train_projects_full):
    u = int(u); v = int(v)
    U2V[u].add(v)
    V2U[v].add(u)

# convert to lists
U2V = [list(s) for s in U2V]
V2U = [list(s) for s in V2U]

print("Avg projects per user (train):", np.mean([len(x) for x in U2V]))
print("Avg users per project (train):", np.mean([len(x) for x in V2U]))
print("Users with 0 projects:", sum(1 for x in U2V if len(x)==0))
print("Projects with 0 users:", sum(1 for x in V2U if len(x)==0))

Avg projects per user (train): 1.3842753886953356
Avg users per project (train): 2.8672518713985675
Users with 0 projects: 4841
Projects with 0 users: 82


In [59]:
# Only compute 2-hop for warm nodes (much faster)
warm_users = np.where(warm_member_mask)[0]
warm_projects = np.where(warm_project_mask)[0]

In [61]:
# User 2-hop teammates
U2U2 = [set() for _ in range(num_members_all)]
for u in warm_users:
    for v in U2V[u]:
        for w in V2U[v]:
            if w != u:
                U2U2[u].add(w)
U2U2 = [list(s) for s in U2U2]
print("Avg 2-hop users per user:", np.mean([len(U2U2[u]) for u in warm_users]) if len(warm_users)>0 else 0)

Avg 2-hop users per user: 2.8951548139555636


In [62]:
# Project 2-hop neighbors
V2V2 = [set() for _ in range(num_projects_all)]
for v in warm_projects:
    for u in V2U[v]:
        for w in U2V[u]:
            if w != v:
                V2V2[v].add(w)
V2V2 = [list(s) for s in V2V2]
print("Avg 2-hop projects per project:", np.mean([len(V2V2[v]) for v in warm_projects]) if len(warm_projects)>0 else 0)

Avg 2-hop projects per project: 3.298317736788013


In [63]:
# Sampling / capping
MAX_U1 = 70
MAX_U2 = 70
MAX_V1 = 70
MAX_V2 = 70

In [64]:
def cap_list(lst, k):
    if len(lst) <= k:
        return lst
    return random.sample(lst, k)

In [65]:
U2V_cap  = [cap_list(x, MAX_U1) for x in U2V]
U2U2_cap = [cap_list(x, MAX_U2) for x in U2U2]
V2U_cap  = [cap_list(x, MAX_V1) for x in V2U]
V2V2_cap = [cap_list(x, MAX_V2) for x in V2V2]

## Two-Hop Functions

In [66]:
class QueryAdditiveAttention(nn.Module):
    """
    Query-based additive attention:
      query: [B, D]
      keys/values X: [B, L, D]
      mask: [B, L] (True = valid)
    returns:
      ctx: [B, D]
    """
    def __init__(self, d_model: int, hidden: int = 128, dropout: float = 0.0):
        super().__init__()
        self.Wq = nn.Linear(d_model, hidden, bias=True)
        self.Wk = nn.Linear(d_model, hidden, bias=True)
        self.v  = nn.Linear(hidden, 1, bias=False)
        self.drop = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, X: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # query: [B,D], X: [B,L,D], mask: [B,L]
        mask = mask.bool()
        B, L, D = X.shape

        # if a row has no neighbors, return zeros
        valid_row = mask.any(dim=1)  # [B]
        out = torch.zeros((B, D), device=X.device, dtype=X.dtype)
        if not valid_row.any():
            return out

        q = query[valid_row]                 # [Bv,D]
        Xv = X[valid_row]                    # [Bv,L,D]
        mv = mask[valid_row]                 # [Bv,L]

        # additive attention score
        # score_{i,j} = v^T tanh(Wq q_i + Wk x_{i,j})
        qh = self.Wq(q).unsqueeze(1)         # [Bv,1,H]
        kh = self.Wk(Xv)                     # [Bv,L,H]
        e  = self.v(torch.tanh(qh + kh)).squeeze(-1)  # [Bv,L]
        e = self.drop(e)

        neg_inf = torch.finfo(e.dtype).min
        e = e.masked_fill(~mv, neg_inf)
        a = torch.softmax(e, dim=1)          # [Bv,L]

        ctx = torch.bmm(a.unsqueeze(1), Xv).squeeze(1)  # [Bv,D]
        out[valid_row] = ctx
        return out

In [67]:
def build_neighbor_matrix(center_indices, neigh_lists, max_len, device=None, sample_if_long=True):
    """
    center_indices: iterable of center node indices length B
    neigh_lists: list[list[int]] neighbors per node
    returns:
      mat:  [B, max_len] long
      mask: [B, max_len] bool
    """
    device = device or "cpu"
    B = len(center_indices)

    mat  = torch.zeros((B, max_len), dtype=torch.long, device=device)
    mask = torch.zeros((B, max_len), dtype=torch.bool, device=device)

    for i, idx in enumerate(center_indices):
        idx = int(idx)
        if idx < 0 or idx >= len(neigh_lists):
            continue

        neigh = neigh_lists[idx]
        if not neigh:
            continue

        if len(neigh) > max_len:
            neigh = random.sample(neigh, max_len) if sample_if_long else neigh[:max_len]

        mat[i, :len(neigh)] = torch.tensor(neigh, dtype=torch.long, device=device)
        mask[i, :len(neigh)] = True

    return mat, mask

In [68]:
class TwoHopCsdRecStatic(nn.Module):
    """
    Two-hop stage:
      base embeddings are frozen (from your improved one-hop that supports warm+cold in full space).
      trainable parts:
        - attention aggregators for 1-hop and 2-hop
        - fusion layers
    """
    def __init__(self, onehop_user_embs, onehop_proj_embs, common_dim=256, attn_hidden=128, dropout=0.1):
        super().__init__()

        # Base frozen embeddings (MUST be full-space aligned indices)
        self.base_user_lookup = nn.Embedding.from_pretrained(
            torch.from_numpy(onehop_user_embs).float(),
            freeze=True
        )
        self.base_proj_lookup = nn.Embedding.from_pretrained(
            torch.from_numpy(onehop_proj_embs).float(),
            freeze=True
        )

        self.common_dim = common_dim

        # Trainable query-attention aggregators
        self.user_attn_1 = QueryAdditiveAttention(common_dim, attn_hidden, dropout=dropout)  # U -> V -> ctx1 (projects)
        self.user_attn_2 = QueryAdditiveAttention(common_dim, attn_hidden, dropout=dropout)  # U -> U2 -> ctx2 (users)

        self.proj_attn_1 = QueryAdditiveAttention(common_dim, attn_hidden, dropout=dropout)  # V -> U -> ctx1 (users)
        self.proj_attn_2 = QueryAdditiveAttention(common_dim, attn_hidden, dropout=dropout)  # V -> V2 -> ctx2 (projects)

        # Fusion
        self.user_fuse = nn.Linear(common_dim * 3, common_dim)
        self.proj_fuse = nn.Linear(common_dim * 3, common_dim)

    def embed_users_base(self, u_idx: torch.Tensor) -> torch.Tensor:
        return self.base_user_lookup(u_idx)

    def embed_projects_base(self, v_idx: torch.Tensor) -> torch.Tensor:
        return self.base_proj_lookup(v_idx)

    def fuse(self, u_base, v_base, u_ctx1, u_ctx2, v_ctx1, v_ctx2):
        u_final = torch.tanh(self.user_fuse(torch.cat([u_base, u_ctx1, u_ctx2], dim=1)))
        v_final = torch.tanh(self.proj_fuse(torch.cat([v_base, v_ctx1, v_ctx2], dim=1)))
        return u_final, v_final

    def score_vectors(self, u_vec, v_vec):
        return (u_vec * v_vec).sum(dim=1)

    def forward(self,
                u_idx, v_idx,
                # neighbor matrices (already padded)
                u1_mat, u1_mask,   # u -> 1-hop projects indices
                u2_mat, u2_mask,   # u -> 2-hop users indices
                v1_mat, v1_mask,   # v -> 1-hop users indices
                v2_mat, v2_mask):  # v -> 2-hop projects indices

        # base
        u_base = self.embed_users_base(u_idx)     # [B,D]
        v_base = self.embed_projects_base(v_idx)  # [B,D]

        # neighbor embeddings
        # user side:
        u1_emb = self.base_proj_lookup(u1_mat)    # [B,L1,D] (projects)
        u2_emb = self.base_user_lookup(u2_mat)    # [B,L2,D] (users)

        # project side:
        v1_emb = self.base_user_lookup(v1_mat)    # [B,L1,D] (users)
        v2_emb = self.base_proj_lookup(v2_mat)    # [B,L2,D] (projects)

        # query-attn aggregation
        u_ctx1 = self.user_attn_1(u_base, u1_emb, u1_mask)  # [B,D]
        u_ctx2 = self.user_attn_2(u_base, u2_emb, u2_mask)  # [B,D]

        v_ctx1 = self.proj_attn_1(v_base, v1_emb, v1_mask)  # [B,D]
        v_ctx2 = self.proj_attn_2(v_base, v2_emb, v2_mask)  # [B,D]

        # fuse + score
        u_final, v_final = self.fuse(u_base, v_base, u_ctx1, u_ctx2, v_ctx1, v_ctx2)
        return self.score_vectors(u_final, v_final)

In [71]:
twohop = TwoHopCsdRecStatic(
    onehop_user_embs=member_base_emb,     # FULL space user embeddings
    onehop_proj_embs=project_base_emb,    # FULL space project embeddings
    common_dim=256
).to(device)

In [72]:
optimizer = torch.optim.Adam(
    [p for p in twohop.parameters() if p.requires_grad],
    lr=2e-4,
    weight_decay=1e-5
)
# optimizer = torch.optim.Adam(twohop.parameters(), lr=2e-4, weight_decay=1e-5)

In [73]:
def bpr_loss(pos_scores, neg_scores):
    return -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

In [76]:
# Build fast positives by PROJECT from TRAIN ONLY
pos_set_by_project = [set() for _ in range(num_projects_all)]
for u, v in zip(train_members_full, train_projects_full):
    pos_set_by_project[int(v)].add(int(u))

In [80]:
def sample_negative_dev(v_idx: int, max_tries: int = 1000) -> int:
    """Sample a negative developer index not in project's TRAIN positives."""
    pos_set = pos_set_by_project[v_idx]
    if len(pos_set) >= num_members_all:
        return random.randrange(num_members_all)

    for _ in range(max_tries):
        u = random.randrange(num_members_all)
        if u not in pos_set:
            return u

    candidates = list(set(range(num_members_all)) - pos_set)
    return random.choice(candidates) if candidates else random.randrange(num_members_all)

In [83]:
BATCH_SIZE = 128
EPOCHS = 8

train_pairs = list(zip(train_members_full.tolist(), train_projects_full.tolist()))

for epoch in range(1, EPOCHS + 1):
    random.shuffle(train_pairs)
    twohop.train()

    total_loss = 0.0
    num_batches = (len(train_pairs) + BATCH_SIZE - 1) // BATCH_SIZE
    pbar = tqdm(range(num_batches), desc=f"TwoHop Epoch {epoch}/{EPOCHS}")

    for bi in pbar:
        batch = train_pairs[bi * BATCH_SIZE : (bi + 1) * BATCH_SIZE]
        if not batch:
            continue

        optimizer.zero_grad(set_to_none=True)

        # ---- Fix project v, rank developers ----
        v_list    = [int(x[1]) for x in batch]
        u_poslist = [int(x[0]) for x in batch]
        u_neglist = [sample_negative_dev(v) for v in v_list]

        v_idx = torch.tensor(v_list, dtype=torch.long, device=device)
        u_pos = torch.tensor(u_poslist, dtype=torch.long, device=device)
        u_neg = torch.tensor(u_neglist, dtype=torch.long, device=device)
        B = len(v_list)

        # ---- base embeddings ----
        v_base    = twohop.embed_projects_base(v_idx)  # [B,D]
        upos_base = twohop.embed_users_base(u_pos)     # [B,D]
        uneg_base = twohop.embed_users_base(u_neg)     # [B,D]

        # ---- neighbor matrices (build on CPU, then move) ----
        v1_mat, v1_mask = build_neighbor_matrix(v_list, V2U_cap,  MAX_V1)
        v2_mat, v2_mask = build_neighbor_matrix(v_list, V2V2_cap, MAX_V2)

        up1_mat, up1_mask = build_neighbor_matrix(u_poslist, U2V_cap,  MAX_U1)
        up2_mat, up2_mask = build_neighbor_matrix(u_poslist, U2U2_cap, MAX_U2)

        un1_mat, un1_mask = build_neighbor_matrix(u_neglist, U2V_cap,  MAX_U1)
        un2_mat, un2_mask = build_neighbor_matrix(u_neglist, U2U2_cap, MAX_U2)

        v1_mat, v1_mask = v1_mat.to(device), v1_mask.to(device)
        v2_mat, v2_mask = v2_mat.to(device), v2_mask.to(device)

        up1_mat, up1_mask = up1_mat.to(device), up1_mask.to(device)
        up2_mat, up2_mask = up2_mat.to(device), up2_mask.to(device)

        un1_mat, un1_mask = un1_mat.to(device), un1_mask.to(device)
        un2_mat, un2_mask = un2_mat.to(device), un2_mask.to(device)

        # ---- embed project neighbors ----
        # v1: users, v2: projects
        v1_user_emb = twohop.base_user_lookup(v1_mat)  # [B,MAX_V1,D]
        v2_proj_emb = twohop.base_proj_lookup(v2_mat)  # [B,MAX_V2,D]

        v_ctx1 = twohop.proj_attn_1(v_base, v1_user_emb, v1_mask)
        v_ctx2 = twohop.proj_attn_2(v_base, v2_proj_emb, v2_mask)

        # ---- embed positive user neighbors ----
        up1_proj_emb = twohop.base_proj_lookup(up1_mat)  # [B,MAX_U1,D]
        up2_user_emb = twohop.base_user_lookup(up2_mat)  # [B,MAX_U2,D]

        up_ctx1 = twohop.user_attn_1(upos_base, up1_proj_emb, up1_mask)
        up_ctx2 = twohop.user_attn_2(upos_base, up2_user_emb, up2_mask)

        # ---- embed negative user neighbors ----
        un1_proj_emb = twohop.base_proj_lookup(un1_mat)
        un2_user_emb = twohop.base_user_lookup(un2_mat)

        un_ctx1 = twohop.user_attn_1(uneg_base, un1_proj_emb, un1_mask)
        un_ctx2 = twohop.user_attn_2(uneg_base, un2_user_emb, un2_mask)

        # ---- fuse to final vectors ----
        # IMPORTANT: use the SAME project contexts for pos and neg
        u_pos_final, v_final = twohop.fuse(upos_base, v_base, up_ctx1, up_ctx2, v_ctx1, v_ctx2)
        u_neg_final, _       = twohop.fuse(uneg_base, v_base, un_ctx1, un_ctx2, v_ctx1, v_ctx2)

        # ---- scores + BPR ----
        pos_scores = twohop.score_vectors(u_pos_final, v_final)
        neg_scores = twohop.score_vectors(u_neg_final, v_final)

        loss = bpr_loss(pos_scores, neg_scores)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(twohop.parameters(), 1.0)
        optimizer.step()

        total_loss += float(loss.item())
        pbar.set_postfix(loss=total_loss / (bi + 1))

    print(f"TwoHop Epoch {epoch} avg loss: {total_loss / max(1, num_batches):.4f}")

torch.save(twohop.state_dict(), "/kaggle/working/training_twohop_csdrec_bpr.pt")
print("Saved trained two-hop model ✅")

TwoHop Epoch 1/8: 100%|██████████| 416/416 [00:13<00:00, 31.53it/s, loss=0.103]


TwoHop Epoch 1 avg loss: 0.1033


TwoHop Epoch 2/8: 100%|██████████| 416/416 [00:12<00:00, 32.27it/s, loss=0.0173]


TwoHop Epoch 2 avg loss: 0.0173


TwoHop Epoch 3/8: 100%|██████████| 416/416 [00:12<00:00, 32.42it/s, loss=0.00872]


...

TwoHop Epoch 6 avg loss: 0.0028


TwoHop Epoch 7/8: 100%|██████████| 416/416 [00:12<00:00, 32.36it/s, loss=0.00221]


TwoHop Epoch 7 avg loss: 0.0022


TwoHop Epoch 8/8: 100%|██████████| 416/416 [00:12<00:00, 32.50it/s, loss=0.0018] 


TwoHop Epoch 8 avg loss: 0.0018
Saved trained two-hop model ✅


## save trained data

In [85]:
twohop.eval()

# --- TRAIN internal indices (full space or train-space, depending on your setup) ---
train_user_idx = np.unique(train_members_full).astype(int)      # if you are using full-space indices
train_proj_idx = np.unique(train_projects_full).astype(int)

# save their ORIGINAL ids in same order
train_user_ids = np.array([member_ids_list[i] for i in train_user_idx], dtype=np.int64)
train_proj_ids = np.array([project_ids_list[i] for i in train_proj_idx], dtype=np.int64)

@torch.no_grad()
def save_twohop_user_embeddings(path_npy, path_csv, idx_array, id_array):
    chunks = []
    for start in tqdm(range(0, len(idx_array), BATCH_SIZE), desc="Saving 2-hop user embs"):
        batch_idx = idx_array[start:start+BATCH_SIZE]
        u_idx = torch.tensor(batch_idx, dtype=torch.long, device=device)
        B = len(batch_idx)

        # base
        u_base = twohop.embed_users_base(u_idx)  # [B,D]

        # neighbors
        u1_mat, u1_mask = build_neighbor_matrix(batch_idx, U2V_cap,  MAX_U1, device=device)
        u2_mat, u2_mask = build_neighbor_matrix(batch_idx, U2U2_cap, MAX_U2, device=device)

        # neighbor embeddings
        u1_emb = twohop.base_proj_lookup(u1_mat)  # [B,L1,D]
        u2_emb = twohop.base_user_lookup(u2_mat)  # [B,L2,D]

        # contexts
        u_ctx1 = twohop.user_attn_1(u_base, u1_emb, u1_mask)
        u_ctx2 = twohop.user_attn_2(u_base, u2_emb, u2_mask)

        # fuse -> final
        u_final = torch.tanh(twohop.user_fuse(torch.cat([u_base, u_ctx1, u_ctx2], dim=1)))  # [B,D]
        chunks.append(u_final.cpu())

    embs = torch.cat(chunks, dim=0).numpy()
    np.save(path_npy, embs)
    pd.DataFrame({"member_id": id_array}).to_csv(path_csv, index=False)
    print("Saved users:", embs.shape, "->", path_npy)

@torch.no_grad()
def save_twohop_project_embeddings(path_npy, path_csv, idx_array, id_array):
    chunks = []
    for start in tqdm(range(0, len(idx_array), BATCH_SIZE), desc="Saving 2-hop project embs"):
        batch_idx = idx_array[start:start+BATCH_SIZE]
        v_idx = torch.tensor(batch_idx, dtype=torch.long, device=device)
        B = len(batch_idx)

        # base
        v_base = twohop.embed_projects_base(v_idx)  # [B,D]

        # neighbors
        v1_mat, v1_mask = build_neighbor_matrix(batch_idx, V2U_cap,  MAX_V1, device=device)
        v2_mat, v2_mask = build_neighbor_matrix(batch_idx, V2V2_cap, MAX_V2, device=device)

        # neighbor embeddings
        v1_emb = twohop.base_user_lookup(v1_mat)  # [B,L1,D]
        v2_emb = twohop.base_proj_lookup(v2_mat)  # [B,L2,D]

        # contexts
        v_ctx1 = twohop.proj_attn_1(v_base, v1_emb, v1_mask)
        v_ctx2 = twohop.proj_attn_2(v_base, v2_emb, v2_mask)

        # fuse -> final
        v_final = torch.tanh(twohop.proj_fuse(torch.cat([v_base, v_ctx1, v_ctx2], dim=1)))  # [B,D]
        chunks.append(v_final.cpu())

    embs = torch.cat(chunks, dim=0).numpy()
    np.save(path_npy, embs)
    pd.DataFrame({"project_id": id_array}).to_csv(path_csv, index=False)
    print("Saved projects:", embs.shape, "->", path_npy)

In [86]:
save_twohop_user_embeddings(
    "/kaggle/working/twohop_train_member_embeddings.npy",
    "/kaggle/working/twohop_train_member_ids.csv",
    train_user_idx,
    train_user_ids
)

save_twohop_project_embeddings(
    "/kaggle/working/twohop_train_project_embeddings.npy",
    "/kaggle/working/twohop_train_project_ids.csv",
    train_proj_idx,
    train_proj_ids
)

Saving 2-hop user embs: 100%|██████████| 263/263 [00:04<00:00, 56.51it/s]


Saved users: (33621, 256) -> /kaggle/working/twohop_train_member_embeddings.npy


Saving 2-hop project embs: 100%|██████████| 145/145 [00:02<00:00, 60.52it/s]

Saved projects: (18487, 256) -> /kaggle/working/twohop_train_project_embeddings.npy


## Evaluation

In [87]:
def rank_metrics(ranks, K=10):
    """
    ranks: list of 1-based ranks of the true developer (1 means best).
    """
    hit = np.mean([1.0 if r <= K else 0.0 for r in ranks])
    ndcg = np.mean([1.0 / math.log2(r + 1) if r <= K else 0.0 for r in ranks])
    return hit, ndcg

In [88]:
K = 10
NUM_NEG = 99

# Build positive sets using TRAIN/TEST (FULL space)
train_pos_by_project = [set() for _ in range(num_projects_all)]
for u, v in zip(train_members_full, train_projects_full):
    train_pos_by_project[int(v)].add(int(u))

test_pos_by_project = [set() for _ in range(num_projects_all)]
for u, v in zip(test_members_full, test_projects_full):
    test_pos_by_project[int(v)].add(int(u))


def sample_eval_candidates_users(v_idx, u_true, num_neg=99):
    """
    For a fixed project v_idx, return [u_true] + num_neg negatives,
    where negatives are not in TRAIN/TEST positives of that project.
    """
    v_idx = int(v_idx)
    u_true = int(u_true)

    forbid = train_pos_by_project[v_idx] | test_pos_by_project[v_idx]

    candidates = [u_true]
    seen = {u_true}

    while len(candidates) < 1 + num_neg:
        u = np.random.randint(0, num_members_all)
        if (u in seen) or (u in forbid):
            continue
        candidates.append(u)
        seen.add(u)

    return candidates


true_ranks = []
twohop.eval()

with torch.no_grad():
    for (u_true, v) in tqdm(list(zip(test_members_full, test_projects_full)), desc="TwoHop Evaluation"):

        # 1) candidate developers (true at index 0)
        cand_users = sample_eval_candidates_users(v, u_true, num_neg=NUM_NEG)
        B = len(cand_users)  # = 1 + NUM_NEG

        # 2) tensors
        u_idx = torch.tensor(cand_users, dtype=torch.long, device=device)
        v_idx = torch.tensor([int(v)] * B, dtype=torch.long, device=device)

        # 3) build neighbor matrices for USER candidates
        u1_mat, u1_mask = build_neighbor_matrix(cand_users, U2V_cap,  MAX_U1, device=device)
        u2_mat, u2_mask = build_neighbor_matrix(cand_users, U2U2_cap, MAX_U2, device=device)

        # 4) build neighbor matrices for PROJECT (repeated B times)
        v_list = [int(v)] * B
        v1_mat, v1_mask = build_neighbor_matrix(v_list, V2U_cap,  MAX_V1, device=device)
        v2_mat, v2_mask = build_neighbor_matrix(v_list, V2V2_cap, MAX_V2, device=device)

        # 5) two-hop scores (project->dev ranking, but score is symmetric dot(u_final, v_final))
        scores = twohop(
            u_idx, v_idx,
            u1_mat, u1_mask,
            u2_mat, u2_mask,
            v1_mat, v1_mask,
            v2_mat, v2_mask
        ).detach().cpu().numpy()

        # rank: true developer is at index 0 in cand_users
        order = np.argsort(-scores)
        rank = int(np.where(order == 0)[0][0]) + 1
        true_ranks.append(rank)

# Final metrics
hit10, ndcg10 = rank_metrics(true_ranks, K=K)

print(f"Candidates: 1+{NUM_NEG} | Evaluated: {len(true_ranks)}")
print(f"Hit@{K}:  {hit10:.4f}")
print(f"NDCG@{K}: {ndcg10:.4f}")
print("Mean rank:", np.mean(true_ranks))
print("Median rank:", np.median(true_ranks))

TwoHop Evaluation: 100%|██████████| 13311/13311 [05:39<00:00, 39.19it/s]

Candidates: 1+99 | Evaluated: 13311
Hit@10:  0.4120
NDCG@10: 0.3252
Mean rank: 27.42348433626324
Median rank: 21.0
